# Ablation функции потерь

Проверяем две гипотезы о текущей цели `bce + dice + 0.3*cls + 0.4*aux`:

1. **Маленькая маска тонет в BCE.** Правка на 0.5% кадра даёт 0.5% слагаемых,
   и градиент по ней теряется на фоне (армы L1, L2, L3).
2. **Цель не совпадает с метрикой.** AIC берёт Dice только по позитивам, а
   негативный кадр не штрафует вообще, пока его маска меньше 1% кадра. Dice же
   на негативе даёт почти полный штраф за любое пятно (армы L4, L5).

| арм | цель | что меняется относительно контроля |
| --- | --- | --- |
| L0 | `bce_dice` | контроль, формула не меняется |
| L1 | `balanced_bce_dice` | вес позитивных пикселей внутри кадра, до 20x |
| L2 | `focal_dice` | focal-перевес BCE, gamma=2, масштаб слагаемого сохранён |
| L3 | `focal_tversky` | Dice -> Tversky (alpha=0.3, beta=0.7, gamma=0.75) |
| L4 | `aic_surrogate` | Dice только по позитивам, негативы — мягкий FPR с порогом 1% |
| L5 | `aic_harmonic` | свёртка тоже как в метрике: гармоническое среднее по батчу |

Всё остальное у армов одинаковое: `pvt_v2_b2`, fold 0, seed 42, та же
аугментация и та же валидация на исходном размере. Бюджет урезан вдвое,
`epoch_size: 12000` вместо 24000 — 96 000 показов на арм. Поэтому **сравнивать
эти AIC с завершёнными полными прогонами нельзя**: контроль L0 нужен именно
как база на том же бюджете. Победивший арм переносится в конфиг с полным
бюджетом и новым `run_name`.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.losses import list_losses
from src.training.ablation import LossAblation

list_losses()

## Выбор армов

`data_path` берётся из конфигов; на другой машине путь до датасета
переопределяется здесь одной строкой, а не правкой шести файлов.

In [ ]:
ARMS = [
    "loss_l0_baseline",
    "loss_l1_pos_weight",
    "loss_l2_focal",
    "loss_l3_focal_tversky",
    "loss_l4_aic_surrogate",
    "loss_l5_aic_harmonic",
]

DATA_PATH = None  # например "/workspace/data" на удалённой машине

ablation = LossAblation([project_root / "configs" / f"{arm}.yaml" for arm in ARMS],
                        data_path=DATA_PATH)
for config in ablation.configs:
    print(f"{config.paths.run_name:36s} {config.loss.name:18s} {config.loss.kwargs}")

## Проверка перед запуском

Один шаг на синтетическом батче: цель считается, градиент доходит до обеих
голов. Дешевле, чем узнать про опечатку в `loss.kwargs` через полчаса
обучения.

In [ ]:
import torch

from src.losses import build_loss

for config in ablation.configs:
    loss_fn = build_loss(config)
    logits = torch.randn(4, 1, 32, 32, requires_grad=True)
    mask = torch.zeros(4, 1, 32, 32)
    mask[:2, :, 2:6, 2:6] = 1.0  # два позитива с маской на 1.5% кадра, два негатива
    out = {"logits": logits, "aux_logits": logits, "cls_logits": torch.randn(4, 1)}
    batch = {"mask": mask, "label": (mask.flatten(1).sum(1, keepdim=True) > 0).float()}
    result = loss_fn(out, batch)
    result.total.backward()
    print(f"{config.loss.name:18s} loss={result.total.item():7.4f} |grad|={logits.grad.abs().sum():8.3f}")

## Запуск серии

Армы идут подряд. Упавший арм не останавливает серию — причина попадёт в
таблицу. Уже досчитанные прогоны пропускаются, поэтому ячейку можно перезапустить
после перезапуска ядра; чтобы пересчитать всё заново, поставьте
`skip_completed=False` и смените `run_name` в конфигах.

In [ ]:
results = ablation.run()
LossAblation.compare(results)

## Разбор

`delta_vs_control` — разница AIC с армом L0 на том же бюджете. Смотреть надо не
только на AIC: арм может выиграть Dice и проиграть FPR, и тогда его стоит
пробовать в паре с другим порогом `cls`, а не отбрасывать.

In [ ]:
table = LossAblation.compare(results)
table[["arm", "loss", "aic", "dice_pos", "fpr_neg", "mask_threshold", "cls_threshold"]]

In [ ]:
import pandas as pd

# Кривая обучения по армам: не выиграл ли арм просто за счёт более быстрого старта.
from src.training.runs import Run

history = pd.concat([
    Run.open(config.paths.runs_path / config.paths.run_name).history.assign(
        arm=config.paths.run_name.split("_loss_")[-1])
    for config in ablation.configs
    if (config.paths.runs_path / config.paths.run_name / "metrics.jsonl").exists()
])
history.pivot_table(index="epoch", columns="arm", values="val/aic_tuned")